# Week 12

Classification networks and training strategies.

In [ ]:
!wget -q https://github.com/PSAM-5020-2025F-A/5020-utils/raw/main/src/data_utils.py
!wget -q https://github.com/PSAM-5020-2025F-A/5020-utils/raw/main/src/image_utils.py
!wget -q https://github.com/PSAM-5020-2025F-A/5020-utils/raw/main/src/nn_utils.py

!wget -qO- https://github.com/PSAM-5020-2025F-A/5020-utils/releases/latest/download/lfw.tar.gz | tar xz

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from torch import nn, Tensor
from torch.optim import SGD

from data_utils import LFWUtils, classification_error, display_confusion_matrix
from image_utils import make_image
from nn_utils import get_labels, get_num_params

## Neural Network Review

We saw how to do regression using neural networks in the [WK11](https://github.com/PSAM-5020-2026S-A/WK11) notebook. This notebook shows what to modify to turn our network into a classification network, while introducing some other types of layers that help training accuracy and speed.

The steps for training a neural network are similar to all of the ML processes we've seen so far:
- Data split
- Data pre-preprocessing
- Training/Fitting
- Validation

The specific steps for setting up our classification training will be:

- Load dataset and do any kind of pre-pre-processing (like get pixels and labels for each image)
- Split data into train/test datasets
- Perform any kind of pre-processing (like scaling or encoding)
- Load features and labels into `Tensors`
- Build a NN model
- Set up an optimizer
- Pick a cost/loss function
- Implement an evaluation function and any other kind of visualization that helps quantify/evaluate the model
- Train model
- Evaluate

## Neural Networks and Classification

The modifications to our network aren't too hard. We first need to have as many output nodes as we have classes in our dataset. If we're trying to classify images into $25$ categories, we need $25$ output neurons.

We also need to use a different kind of loss/cost function that will force the network to only have one activated neuron in its output layer, per prediction. If we give it data for something that should be classified as class $7$, the neuron that represents class $7$ at the output layer should be activated way more than the others.

### Cross Entropy Loss

The cost of being wrong for a classification network is calculated using a function called *Cross Entropy*. It basically measures how different two distributions are, and in this case, we want to compare the distributions of values from the last layer of our network and what we expected to get at the last layer, over all of the input rows that it was shown.

For example, let's say we are training a classification neural network on a dataset with $3$ classes. The correct distribution of labels in the training dataset is:

| $\text{class}$ | $\text{distribution}$ |
|----|----|
| $0$ | $45\%$ |
| $1$ | $35\%$ |
| $2$ | $20\%$ |

but instead we get the following distribution for the predictions:

| $\text{class}$ | $\text{distribution}$ |
|----|----|
| $0$ | $32\%$ |
| $1$ | $32\%$ |
| $2$ | $36\%$ |

*Cross Entropy* is a way to quantify how different these two distributions are and, hopefully, guide our network parameters to make them more similar.

The actual cost calculation is a sum of how different each prediction is from a "perfect" prediction. For example, if we're trying to classify something as class $0$, we should get output neuron activations that are something like: `[1.0, 0.0, 0.0]`. If instead we get: `[0.5, 0.8, 0.1]`, the *Cross Entropy* contribution for this sample is calculated as:
$$\displaystyle - \log\left(\frac{e^{0.5}}{e^{0.5} + e^{0.8} + e^{0.1}}\right)$$

By using this as the cost function, the neural network can optimize its parameters to increase the value of the first output, while decreasing the values of the other outputs. It doesn't have to get `[1.0, 0.0, 0.0]` exactly, so the network can optimize its parameters to just make the first output as different as possible from the others.

We'll use the built in [`CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) function in the `PyTorch` library.

### Load and split Dataset

We'll use the _Labeled Faces in the Wild_ dataset from the WK11-review notebook.

As always, we start with the data, and this time we'll put the pixel and label information straight into `Tensor` objects.

The `LFWUtils.train_test_split(0.5)` function gives us some `Python` objects we can use to create our `Tensor` objects.

The `pixels` key gives us a list of the images' pixel data, and the `label` key gives us the images' label IDs.

We don't have to do any normalization since the pixels will be in a known, well-defined, range of $[0 \text{ - } 255]$.

The only thing we have to do differently is cast the label `Tensor` objects to `long`. This is to ensure the numbers in those `Tensor` objects are whole numbers and don't have decimal points.

In [ ]:
train, test = LFWUtils.train_test_split("./data/image/lfw/cropped", 0.5)

display(make_image(train["pixels"][110], LFWUtils.IMAGE_WIDTH))
print(LFWUtils.LABELS[train["labels"][110]])

LFWUtils.IMAGE_SIZE

In [ ]:
train, test = LFWUtils.train_test_split("./data/image/lfw/cropped", 0.5)

x_train = Tensor(train["pixels"])
y_train = Tensor(train["labels"]).long()

x_test = Tensor(test["pixels"])
y_test = Tensor(test["labels"]).long()

print("Dataset Samples")
print("\tTrain:", len(x_train))
print("\tTest:", len(x_test))

print("\nDataset Shape:", list(x_train.shape))
print("\nSample Shape:", list(x_train[0].shape))

### Look at data

We can visualize some of the images, their text labels and label IDs

In [ ]:
idx = 250

id = train["labels"][idx]
print(id, LFWUtils.LABELS[id])
display(make_image(train["pixels"][idx], width=LFWUtils.WIDTH))

In [ ]:
len(LFWUtils.LABELS)

### Model, Optimizer and Cost Function

We'll start with the simplest kind of network again, with just an input and an output layer.

The input layer has as many neurons as the number of pixels in each image, and the output layer has one neuron per possible class.

It looks like this, and is juts like our regression network above, but has more output neurons:

<img src="./imgs/linear_22100x26.jpg" width="800px"/>

Our optimizer will be `SGD` again.

Our cost function is a bit different. Previously, we used $L2$ distances to calculate the root mean square error of our regression predictions and used that value as the cost function for gradient descent.

In order to backpropagate error values through our neurons when doing classification, we have to turn the discrete nature of our labels/classes and their errors into something that has more smooth and integratable values.

That's what the `CrossEntropyLoss()` function does for us. It looks at the outputs of our model and transforms the regression-type continuous values at our neuron outputs into class prediction probabilities in a way that our optimization still works.

The loss for a given prediction is calculated by the equation:
$$-\log\left(\frac{e^{y_c}}{\sum_{i=0}^{C}e^{y_i}}\right)$$

Where, $y_c$ is the value of the output neuron that corresponds to the correct class of the prediction, and $y_i$ are all of the output neuron values. This equation, with its $e^y$ and $\log()$, results in a very high value whenever the correct neuron is not activated, but gives us a $0$ (or a value close to $0$) when the correct output neuron is the most strongly activated neuron amongst a few activated neurons.

More importantly, this equation is integratable and can be used in backpropagation for adjusting our neuron parameters.

### Let's train !

Let's create a single layer neural network, like the first one from class, and train it with the training data.

In addition to the actual model/network, we also need an optimizer and a loss function.

In [ ]:
x_train.shape[1], len(LFWUtils.LABELS)

In [ ]:
# TODO: Create the model and optimizer, the loss function is already defined
model = nn.Sequential(
  nn.Linear(x_train.shape[1], len(LFWUtils.LABELS))
)

optim = SGD(model.parameters(), lr=0.05, momentum=0.9)

loss_fn = nn.CrossEntropyLoss()

### Test Shapes

The `x` and `y` variables below actually hold pixel and label information for $445$ images. We feed all of them into the network at once and predict all of the labels at the same time. We can then calculate loss, backpropagate error information, update model parameters, and repeat.

Before we do that, we're just going to put the inputs through the model to check if the parameter shapes match.

Since the model computes a type of probability for each possible class, our output should have a row for each item in our training dataset and a column for each possible class.

In [ ]:
out = model(x_train)

print("Input shape:", x_train.shape)
print("Output shape:", out.shape)

print("First outcome value:", y_train[idx])
print("First model output:", out[idx])

In [ ]:
LFWUtils.LABELS[14]

### The Loop

Create a training loop like we saw in WK11.

This loop should:
- Predict classes by feeding all of the inputs into the `model`
- Measure `loss` (this is just `loss_fn(predicted, actual)`)
- Get the optimizer to back-propagate and annotate the neurons
- Update parameters

The loop should be repeated as long as the loss keeps improving, and it doesn't look like the model is overfitting with the training data.

In order to check if the model is overfitting, we can sporadically run evaluations within the training loop in order to see if the model performs similarly with `train` and `test` data.

But ! Our network actually outputs a series of values for each image that we give it. In order to determine the exact class number of its predictions, we have to find the index of the output neuron with the largest value, which is an operation called `argmax()` (similar to `argsort()` from week 10).

It's not hard to do this manually, but we can use the `get_labels(model, inputs)` function inside the `nn_utils` file to run our `model` on all of the data in a given dataset and return the predicted labels for all of the samples.

In [ ]:
get_num_params(model)

In [ ]:
# TODO: iterate epochs
# TODO: predict
# TODO: measure loss
# TODO: compute gradient and step optimizer
# TODO: show progress

for e in range(32):
  optim.zero_grad()
  preds = model(x_train)
  loss = loss_fn(preds, y_train)
  loss.backward()
  optim.step()

  if e%8 == 0:
    train_preds = model(x_train)
    train_preds_ids = train_preds.argmax(dim=1)
    train_classification_error = classification_error(y_train, train_preds_ids)

    test_preds = model(x_test)
    test_preds_ids = test_preds.argmax(dim=1)
    test_classification_error = classification_error(y_test, test_preds_ids)

    print("training loss:", loss.item(), "training err:", train_classification_error, "test err:", test_classification_error)

### Interpretation

The loss/cost value can oscillate up and down a bit, but, overall should steadily decrease. This oscillation has to do with the `SGD` optimizer and how it makes some decisions that it sometimes has to undo, but overall, the model looks like it's learning.

Do we want to keep running this cell until the loss gets really small?

How low can we get our test error ?

In [ ]:
train_preds = model(x_train)
train_preds_ids = train_preds.argmax(dim=1)
# train_preds_ids
y_train

### Add Layers

Maybe we can improve our classification by adding some hidden layers.

We might want to do this:

In [ ]:
model =  nn.Sequential(
  # nn.Linear(x_train.shape[1], x_train.shape[1] // 2),
  nn.Linear(x_train.shape[1], 2000),
  nn.ReLU(),

  # nn.Linear(x_train.shape[1] // 2, x_train.shape[1] // 16),
  # nn.ReLU(),

  # nn.Linear(x_train.shape[1] // 16, x_train.shape[1] // 32),
  # nn.ReLU(),

  nn.Linear(2000, len(LFWUtils.LABELS)),
)

But, now this neural network has these many parameters:

In [ ]:
get_num_params(model)

That's a lot.

Our regular codespaces computer won't be able to optimize a network of this size in a reasonable amount of time.

We saw in the homework that if we want to add layers we have to reduce the number of our features. So .... PCA.

In [ ]:
len(train["pixels"])

In [ ]:
std = StandardScaler().set_output(transform="pandas")
pca = PCA(n_components=445).set_output(transform="pandas")

x_pca_train = pca.fit_transform(std.fit_transform(train["pixels"]))
x_pca_test = pca.transform(std.transform(test["pixels"]))

print(pca.n_components_, sum(pca.explained_variance_ratio_))

x_train = Tensor(x_pca_train.values)
y_train = Tensor(train["labels"]).long()

x_test = Tensor(x_pca_test.values)
y_test = Tensor(test["labels"]).long()

x_train.shape

Now we can try adding some layers to our network, since the initial number of features has been reduced by a factor of 50.

In [ ]:
model =  nn.Sequential(
  # TODO: Build the neural net here
  nn.Linear(x_train.shape[1], 4*x_train.shape[1]),
  nn.ReLU(),

  nn.Linear(4*x_train.shape[1], x_train.shape[1]),
  nn.ReLU(),

  nn.Linear(x_train.shape[1], len(LFWUtils.LABELS)),
)

optim = SGD(model.parameters(), lr=1e-2, momentum=0.9)

loss_fn = nn.CrossEntropyLoss()

out = model(x_train)

print("Input shape:", x_train.shape)
print("Output shape:", out.shape)
print("Num Params:", get_num_params(model))

In [ ]:
# TRAIN LOOP

for e in range(128):
  # clear slopes in optimizer
  optim.zero_grad()
  # compute labels from model
  labels_pred = model(x_train)
  # calculate how wrong model is
  loss = loss_fn(labels_pred, y_train)
  # compute slopes for cost function
  loss.backward()
  # adjust model's parameters
  optim.step()

  # keep an eye on loss as we train
  if e % 32 == 0:
    train_predictions = get_labels(model, x_train)
    test_predictions = get_labels(model, x_test)
    train_error = classification_error(y_train, train_predictions)
    test_error = classification_error(y_test, test_predictions)
    print(f"Epoch: {e} loss: {loss.item():.4f}, train error: {train_error:.4f}, test error: {test_error:.4f}")

In [ ]:
train_predictions = get_labels(model, x_train)
test_predictions = get_labels(model, x_test)

print("train error:", f"{classification_error(y_train, train_predictions):.4f}")
print("test error", f"{classification_error(y_test, test_predictions):.4f}")

display_confusion_matrix(y_train, train_predictions, display_labels=LFWUtils.LABELS)
display_confusion_matrix(y_test, test_predictions, display_labels=LFWUtils.LABELS)

### Interpretation

The result is mostly the same, which is not surprising.

We did add layers, but the network didn't need any extra neurons to do well on the training data.

It needs help with the testing data, or, another way to say this is: it needs help generalizing without memorizing.

### Make It Harder

Neural network models can seem simple to explain in a general sense: they're long and wide computation graphs made up of simple operations that have been tuned to achieve a specific task. Once they're training, or trained, their details and specificities are a little less easy to describe. It's hard to know exactly what each neuron is doing, and what part of the computation they are responsible for. We can train the same network, with the same parameters, using the same input data, and end up with wildly different results.

This is one reason why it's hard to debug a network when it doesn't seem to be learning properly, or when it starts to overfit and memorize the training data. Which neurons do we tune ?

One common situation that can lead to overfitting is when a network ends up with parameters that make it perform well on the training data without really activating all of its neurons. This is usually what is happening if adding layers to a network doesn't improve its performance.

One set of strategies for improving neural network training in these cases involves making the training process harder than it has to be. It's like we're challenging the neural network to learn more than it has, so that later it has an easier time with the regular data.

#### Dropout

One simple technique to achieve this is to add `Dropout` layers to our network. A `Dropout` layer is a layer of neurons that don't perform any mathematical operation, but are selectively dropped out of the network randomly during training. This has the effect of randomly changing the network's architecture during training and preventing the network from becoming too reliant on specific neurons. Instead, it encourages the network to learn more robust features by activating more diverse sets of neurons.

<img src="./imgs/dropout.jpg" width="800px"/>

#### Activation Normalization

Another technique that is used to keep our neural networks from memorizing data has to do with the range of the values that get passed between its inner layers.

Input data coming into the network is most likely normalized, but after the first layer, the network weights might really change the distribution of the data as it flows through the network. Moreover, individual batches with different input value distributions can bias the network towards certain goals.

<img src="./imgs/norm_activation.jpg" width="720px"/>

#### Batch Normalization

One way to handle these situations is to normalize the data as it passes through the network. Batch Normalization is the process of normalizing the activations of our network by using the mean and standard deviation of an activation neuron across a batch. The result is that the activations between batches become more similar. Batch normalization is dependent on batch size, so it's not effective for small batches.

<img src="./imgs/norm_batch.jpg" width="720px"/>

#### Layer Normalization

Another form of inner-network normalization can be added to make sure no individual layer overpowers the network with activation values that are too large or too small.

Layer Normalization scales activations using the mean and standard deviation of all activations across a layer. It's effective for sequence models like RNNs and Transformers, and for scenarios with small batch sizes, and doesn't require a large batch to get a good estimate for mean and standard deviation. 

<img src="./imgs/norm_layer.jpg" width="720px"/>

In [ ]:
# NOTE: It's uncommon to have both batch and layer normalization for the same activations.
# NOTE: They usually go between the linear layer and the activation, but BatchNorm can also go after activation.

model =  nn.Sequential(
  # TODO: Experiment adding dropout and normalization to the neural network you created above
  # TODO: other layers
  nn.BatchNorm1d()
)

optim = SGD(model.parameters(), lr=1e-2, momentum=0.9)

loss_fn = nn.CrossEntropyLoss()

out = model(x_train)

print("Input shape:", x_train.shape)
print("Output shape:", out.shape)
print("Num Params:", get_num_params(model))

In [ ]:
# TRAIN LOOP

for e in range(256):
  optim.zero_grad()
  labels_pred = model(x_train)
  loss = loss_fn(labels_pred, y_train)
  loss.backward()
  optim.step()

  if e % 32 == 31:
    train_predictions = get_labels(model, x_train)
    test_predictions = get_labels(model, x_test)
    train_error = classification_error(y_train, train_predictions)
    test_error = classification_error(y_test, test_predictions)
    print(f"Epoch: {e} loss: {loss.item():.4f}, train error: {train_error:.4f}, test error: {test_error:.4f}")

### Interpretation

The train and test eval function diverged, but both keep decreasing, so this might be ok.

In the end, the network seems capable of learning for longer and while the classification error for the test dataset doesn't keep up with the error in the train dataset, it does perform better.

In [ ]:
train_predictions = get_labels(model, x_train)
test_predictions = get_labels(model, x_test)

print("train error", f"{classification_error(y_train, train_predictions):.4f}")
print("test error", f"{classification_error(y_test, test_predictions):.4f}")

display_confusion_matrix(y_train, train_predictions, display_labels=LFWUtils.LABELS)
display_confusion_matrix(y_test, test_predictions, display_labels=LFWUtils.LABELS)